In [10]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import sys
import os

# Add the parent directory so we can import from the 'src' folder
sys.path.append(os.path.abspath(os.path.join('..')))

from src.models.eegnet import EEGNet
from src.data_prep import get_cleaned_epochs


In [11]:
# Use the modular function we created to get the clean data
epochs, event_id = get_cleaned_epochs(subject_id=1)

# Convert MNE epochs to a NumPy array [Trials, Channels, TimePoints]
X = epochs.get_data() 
y = epochs.events[:, -1] - 769  # Normalize labels to start at 0


In [12]:
# Add the '1' dimension for the Convolutional layers
if X.ndim == 3:
    X = X[:, None, :, :]

X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)

# Create the DataLoader for batching
dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True)



In [13]:
# 1. Initialize the model with dynamic dimensions
model = EEGNet(nb_classes=len(event_id), Chans=X.shape[2], Samples=X.shape[-1])

# 2. Setup the "Math" parts
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# 3. The Training Loop
print("Starting training...")
for epoch in range(50):
    model.train()
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()           # Reset the math
        outputs = model(batch_X)        # Make a guess
        loss = criterion(outputs, batch_y) # Calculate the error
        loss.backward()                 # Calculate the fix
        optimizer.step()                # Apply the fix (Learning)
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

print("Training Complete!")



Starting training...


RuntimeError: mat1 and mat2 shapes cannot be multiplied (16x1056 and 1024x4)